# Avance del proyecto — Fase 3
## Núcleo algorítmico, eficiencia e implementación orientada a objetos

**Proyecto:** SIMCE 4° Básico 2025 — Matemática  
**Propósito del notebook:** demostrar, de forma reproducible, la evolución del pipeline validado en F2 hacia una arquitectura modular con POO, recursividad pertinente, pruebas y mediciones de eficiencia.

> **Decisión de arquitectura:** las clases y algoritmos reutilizables **no se definen en las celdas**. Viven en `src/` y este notebook los importa, ejecuta, mide y documenta. Así se evita duplicar implementación y se mantiene una única fuente de verdad.

## 1. Preparación reproducible

La primera celda localiza la raíz del repositorio tanto si el notebook se abre desde `F3/` como desde la raíz. También fija las rutas y muestra las versiones relevantes. No se usan rutas absolutas personales.

In [32]:
from pathlib import Path
import platform
import subprocess
import sys
import pandas as pd
import numpy as np
from IPython.display import display
from pandas.testing import assert_frame_equal

ACTUAL = Path.cwd().resolve()
if (ACTUAL / "src").exists():
    PROJECT_ROOT = ACTUAL
elif (ACTUAL.parent / "src").exists():
    PROJECT_ROOT = ACTUAL.parent
else:
    raise FileNotFoundError(f"No se encontró la raíz del proyecto desde {ACTUAL}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.configuracion import COLUMNAS_SIMCE, encontrar_raw_dir, buscar_simce
from src.carga import cargar_simce, construir_dim_geografia
from src.poo import (
    TransformadorTipos, TransformadorTextos, TransformadorCategorias,
    TransformadorGeografia, TransformadorEfectividad, PipelineSIMCE,
)
from src.algoritmo import (
    aplanar_recursivo, construir_jerarquia_geografica, validar_geografia_recursiva,
    enriquecer_geografia_merge, enriquecer_geografia_diccionario, medir,
)
from src.auditoria import construir_auditoria
from src.transformacion import construir_cobertura_regional
from src.validacion import validar_dataset_final
from src.arquitectura import tabla_arquitectura

RAW_DIR = encontrar_raw_dir()
SIMCE_FILE = buscar_simce(RAW_DIR)
print("Raíz     :", PROJECT_ROOT)
print("Entrada  :", SIMCE_FILE)
print("Python   :", sys.version.split()[0])
print("pandas   :", pd.__version__)
print("NumPy    :", np.__version__)
print("Sistema  :", platform.platform())

Raíz     : C:\Users\emper\f1_s01_evaluacion_entregable_grupo7
Entrada  : C:\Users\emper\f1_s01_evaluacion_entregable_grupo7\data\raw\simce4b2025_rbd_final.csv
Python   : 3.14.7
pandas   : 3.0.5
NumPy    : 2.5.2
Sistema  : Windows-11-10.0.26200-SP0


## 2. Carga y línea base de F2

F3 **no inventa un pipeline nuevo**. Parte del mismo conjunto y de las mismas reglas ya validadas en F2. La dimensión territorial contiene 346 comunas y se valida antes de usarla.

In [33]:
dim_geo = construir_dim_geografia()
df_raw = cargar_simce(SIMCE_FILE, COLUMNAS_SIMCE)

print(f"SIMCE bruto : {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
print(f"DimGeografia: {len(dim_geo):,} comunas")
assert len(dim_geo) == 346
assert not dim_geo.duplicated(["cod_reg_rbd", "cod_pro_rbd", "cod_com_rbd"]).any()
display(df_raw.head(3))

SIMCE bruto : 7,143 filas x 20 columnas
DimGeografia: 346 comunas


,rbd,dvrbd,nom_rbd,cod_reg_rbd,cod_pro_rbd,cod_com_rbd,cod_deprov_rbd,nom_deprov_rbd,cod_depe1,cod_depe2,cod_grupo,cod_rural_rbd,nalu_mate4b_rbd,prom_mate4b_rbd,marca_mate4b_rbd,noaplica,codigo_bbdd,fecha_bbdd,grado,agno
0,7826,3,COLEGIO SAN MIGUEL,10,101,10102,101,Llanquihue,3,2,3.0,1,65,263.0,NaN,0,v22025,20260622,4b,2025
1,11407,3,ESCUELA BASICA LOS OLIVOS,16,162,16206,161,Diguillín,2,1,1.0,2,3,221.0,NaN,0,v22025,20260622,4b,2025
2,10743,3,INSTITUTO SAN FRANCISCO,13,136,13602,135,Talagante,3,2,2.0,1,25,220.0,NaN,0,v22025,20260622,4b,2025


## 3. Pipeline F3: mismo resultado, arquitectura distinta

Cada transformación tiene una responsabilidad concreta. `PipelineSIMCE` compone los pasos y registra el flujo. Las clases concretas reutilizan las funciones de F2 para conservar la lógica de negocio validada.

In [34]:
pipeline = PipelineSIMCE([
    TransformadorTipos(),
    TransformadorTextos(),
    TransformadorCategorias(),
    TransformadorGeografia(dim_geo),
    TransformadorEfectividad(),
])

df_transformado = pipeline.ejecutar(df_raw)
df_final = pipeline.construir_producto_final(df_transformado)
auditoria, resumen_auditoria, resumen_marcas = construir_auditoria(df_transformado)
cobertura = construir_cobertura_regional(df_transformado)
validaciones = validar_dataset_final(df_final, df_transformado, auditoria, cobertura)

print("Pasos:", " -> ".join(pipeline.pasos))
display(pd.DataFrame(pipeline.registro))
print(f"Producto final: {len(df_final):,} filas x {df_final.shape[1]} columnas")
print(f"Excluidos auditados: {len(auditoria):,}")
print("Validaciones F2/F3:", sum(validaciones.values()), "/", len(validaciones))

Pasos: TransformadorTipos -> TransformadorTextos -> TransformadorCategorias -> TransformadorGeografia -> TransformadorEfectividad


,paso,filas_entrada,filas_salida,columnas_salida
0,TransformadorTipos,7143,7143,20
1,TransformadorTextos,7143,7143,20
2,TransformadorCategorias,7143,7143,24
3,TransformadorGeografia,7143,7143,34
4,TransformadorEfectividad,7143,7143,37


Producto final: 6,524 filas x 30 columnas
Excluidos auditados: 619
Validaciones F2/F3: 14 / 14


### Resultado esperado y continuidad

La refactorización debe preservar el resultado: **7.143** registros brutos, **6.524** efectivos y **619** no efectivos auditados. Si estos conteos cambian sin una decisión documentada, F3 habría alterado una regla de negocio y debe investigarse.

In [35]:
assert len(df_raw) == 7143, "Cambió la fuente respecto de la línea base documentada."
assert len(df_final) == 6524, "La refactorización cambió el número de efectivos."
assert len(auditoria) == 619, "La refactorización cambió la auditoría de excluidos."
print("Continuidad F2 -> F3 verificada.")

Continuidad F2 -> F3 verificada.


## 4. POO: evidencia de los principios

- **Herencia:** los transformadores concretos heredan de `Transformador`.
- **Polimorfismo:** el pipeline invoca `transformar(df)` sin preguntar qué clase concreta está ejecutando.
- **Encapsulamiento:** estado interno como `_pasos`, `_registro` y `_dim_geografia` no se modifica directamente desde el notebook.
- **Alta cohesión:** cada clase tiene una responsabilidad.
- **Bajo acoplamiento:** `PipelineSIMCE` depende del contrato `Transformador`, no de una implementación específica.

In [36]:
arquitectura = tabla_arquitectura()
display(arquitectura)
print("Pasos expuestos como estructura inmutable:", type(pipeline.pasos).__name__)
print("Comunas encapsuladas en TransformadorGeografia:", TransformadorGeografia(dim_geo).comunas_dimension)

,clase,hereda_de,responsabilidad,metodos_publicos,modulo
0,Transformador,ABC,Contrato común para transformaciones del pipel...,transformar,src/poo.py
1,TransformadorTipos,Transformador,Contrato común para transformaciones del pipel...,transformar,src/poo.py
2,TransformadorTextos,Transformador,Contrato común para transformaciones del pipel...,transformar,src/poo.py
3,TransformadorCategorias,Transformador,Contrato común para transformaciones del pipel...,transformar,src/poo.py
4,TransformadorGeografia,Transformador,Encapsula la dimensión geográfica necesaria pa...,transformar,src/poo.py
5,TransformadorEfectividad,Transformador,Contrato común para transformaciones del pipel...,transformar,src/poo.py
6,PipelineSIMCE,object,Compone transformadores con bajo acoplamiento ...,"construir_producto_final, ejecutar",src/poo.py


Pasos expuestos como estructura inmutable: tuple
Comunas encapsuladas en TransformadorGeografia: 346


## 5. Flujo y diseño estructurado

El flujo queda explícito y verificable: **carga → tipos → textos → categorías → geografía → efectividad → auditoría/validación → producto final**. El notebook orquesta; `src` implementa. Esta separación permite reutilizar las mismas piezas en otras fases sin copiar celdas.

In [37]:
flujo = pd.DataFrame({
    "orden": range(1, 9),
    "etapa": ["Carga", "Tipos", "Textos", "Categorías", "Geografía", "Efectividad", "Auditoría/validación", "Producto final"],
    "responsable": ["src/carga.py", "src/poo.py", "src/poo.py", "src/poo.py", "src/poo.py", "src/poo.py", "src/auditoria.py + src/validacion.py", "src/transformacion.py"],
})
display(flujo)

,orden,etapa,responsable
0,1,Carga,src/carga.py
1,2,Tipos,src/poo.py
2,3,Textos,src/poo.py
3,4,Categorías,src/poo.py
4,5,Geografía,src/poo.py
5,6,Efectividad,src/poo.py
6,7,Auditoría/validación,src/auditoria.py + src/validacion.py
7,8,Producto final,src/transformacion.py


## 6. Recursividad aplicada a una estructura real

La recursividad se usa para recorrer la jerarquía **Región → Provincia → Comuna** y para aplanar metadatos anidados. Es pertinente porque la profundidad de una estructura jerárquica puede cambiar; codificar bucles anidados fija artificialmente esa profundidad.

**Caso base:** un valor/hoja que ya no contiene otra estructura.  
**Caso recursivo:** un diccionario o colección que contiene nuevos nodos.  
**Complejidad:** visitar `n` nodos cuesta `O(n)` tiempo y la pila requiere `O(h)`, donde `h` es la profundidad máxima.

In [38]:
jerarquia = construir_jerarquia_geografica(dim_geo)
errores_geo = validar_geografia_recursiva(jerarquia)
comunas_jerarquia = sum(
    len(comunas)
    for provincias in jerarquia.values()
    for comunas in provincias.values()
)

metadatos = {
    "proyecto": {"fase": 3, "asignatura": "Matematica"},
    "datos": {"filas_brutas": len(df_raw), "filas_finales": len(df_final)},
    "geografia": {"regiones": len(jerarquia), "comunas": comunas_jerarquia},
}

print("Errores jerarquía:", errores_geo)
print("Comunas recorridas:", comunas_jerarquia)
assert errores_geo == []
assert comunas_jerarquia == 346
display(pd.DataFrame(aplanar_recursivo(metadatos).items(), columns=["clave", "valor"]))

Errores jerarquía: []
Comunas recorridas: 346


,clave,valor
0,proyecto.fase,3
1,proyecto.asignatura,Matematica
2,datos.filas_brutas,7143
3,datos.filas_finales,6524
4,geografia.regiones,16
5,geografia.comunas,346


## 7. Validación técnica y pruebas reejecutables

Las pruebas viven en `tests/`, no en el notebook. Cubren escenarios normales, límite y de excepción: regla de efectividad, códigos desconocidos, encapsulamiento, polimorfismo, recursividad, equivalencia de algoritmos y detección de una dimensión geográfica que viola `many_to_one`.

In [39]:
resultado_pruebas = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_*.py", "-v"],
    cwd=str(PROJECT_ROOT), capture_output=True, text=True,
)
print(resultado_pruebas.stdout)
print(resultado_pruebas.stderr)
if resultado_pruebas.returncode != 0:
    raise RuntimeError("Las pruebas técnicas de F3 no fueron superadas.")
print("Suite de pruebas superada correctamente.")


test_caso_con_observacion (test_regla_efectividad.TestReglaEfectividad.test_caso_con_observacion) ... ok
test_caso_limite (test_regla_efectividad.TestReglaEfectividad.test_caso_limite) ... ok
test_caso_normal (test_regla_efectividad.TestReglaEfectividad.test_caso_normal) ... ok
test_codigo_desconocido (test_regla_efectividad.TestReglaEfectividad.test_codigo_desconocido) ... ok
test_marca_2_con_puntaje (test_regla_efectividad.TestReglaEfectividad.test_marca_2_con_puntaje) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.010s

OK

Suite de pruebas superada correctamente.


## 8. Eficiencia y optimización

Se compara una operación real del proyecto: enriquecer 7.143 establecimientos con una dimensión de 346 comunas. Primero se exige **equivalencia funcional** y recién después se comparan tiempo y memoria.

Implementaciones:
1. `pandas.merge(..., validate="many_to_one")`: operación vectorizada y validación explícita de cardinalidad.
2. Diccionario con clave compuesta `(región, provincia, comuna)`: búsqueda promedio `O(1)` por fila después de construir el índice.

En términos asintóticos, ambas alternativas son aproximadamente lineales respecto de los datos de entrada, pero las constantes, memoria y garantías de integridad son distintas.

In [40]:
# Se miden datos ya tipados/limpios para aislar el costo del enriquecimiento geográfico.
df_para_geo = TransformadorCategorias().transformar(
    TransformadorTextos().transformar(TransformadorTipos().transformar(df_raw))
)

merge_result = enriquecer_geografia_merge(df_para_geo, dim_geo, validar=True)
dict_result = enriquecer_geografia_diccionario(df_para_geo, dim_geo, validar=True)
assert_frame_equal(
    merge_result.sort_index(axis=1), dict_result.sort_index(axis=1), check_dtype=False
)
print("Equivalencia merge vs diccionario: OK")

Equivalencia merge vs diccionario: OK


### 8.1 Costo/beneficio de `validate="many_to_one"`

La validación añade una comprobación de integridad: impide que una dimensión con claves duplicadas multiplique silenciosamente filas. Se mide su costo relativo y se demuestra su beneficio provocando intencionalmente una clave duplicada.

In [41]:
comparacion_validate = []
for validar in [False, True]:
    m = medir(enriquecer_geografia_merge, df_para_geo, dim_geo, validar=validar, repeticiones=7)
    comparacion_validate.append({
        "validate_many_to_one": validar,
        "tiempo_promedio_s": m["tiempo_promedio_s"],
        "memoria_pico_promedio_mb": m["memoria_pico_promedio_mb"],
    })
comparacion_validate = pd.DataFrame(comparacion_validate)
display(comparacion_validate)

base = float(comparacion_validate.loc[~comparacion_validate["validate_many_to_one"], "tiempo_promedio_s"].iloc[0])
validado = float(comparacion_validate.loc[comparacion_validate["validate_many_to_one"], "tiempo_promedio_s"].iloc[0])
print(f"Sobrecosto temporal observado: {(validado/base - 1)*100:.2f}%")

dim_duplicada = pd.concat([dim_geo, dim_geo.iloc[[0]]], ignore_index=True)
try:
    enriquecer_geografia_merge(df_para_geo, dim_duplicada, validar=True)
except Exception as exc:
    print("Duplicado detectado correctamente:", type(exc).__name__)

,validate_many_to_one,tiempo_promedio_s,memoria_pico_promedio_mb
0,False,0.005519,0.644480
1,True,0.007854,0.645914


Sobrecosto temporal observado: 42.31%
Duplicado detectado correctamente: MergeError


### 8.2 Crecimiento con el tamaño

Para evitar una conclusión basada en una sola medición, se repite el benchmark con distintos tamaños. El objetivo no es extrapolar a millones de filas, sino observar si el costo crece de forma coherente con el análisis de complejidad.

In [42]:
crecimiento = []
for n in [500, 2000, len(df_para_geo)]:
    muestra = df_para_geo.iloc[:n].copy()
    for nombre, funcion in [("merge", enriquecer_geografia_merge), ("diccionario", enriquecer_geografia_diccionario)]:
        m = medir(funcion, muestra, dim_geo, validar=True, repeticiones=3)
        crecimiento.append({"filas": n, "algoritmo": nombre, "tiempo_s": m["tiempo_promedio_s"]})
display(pd.DataFrame(crecimiento))

,filas,algoritmo,tiempo_s
0,500,merge,0.007622
1,500,diccionario,0.039203
2,2000,merge,0.006413
3,2000,diccionario,0.043505
4,7143,merge,0.007951
5,7143,diccionario,0.074564


## 9. Análisis de sensibilidad de la regla de efectividad

Se comparan tres escenarios sin cambiar la regla oficial:
- **A:** solo registros efectivos (producto oficial del pipeline).
- **B:** todos los registros que conservan puntaje.
- **C:** efectivos + registros `marca 2` con puntaje, únicamente como escenario diagnóstico.

El escenario C no reincorpora registros al dataset final; cuantifica cuánto cambiaría el resultado si esos 55 casos se incluyeran.

In [43]:
con_puntaje = df_transformado["prom_mate4b_rbd"].notna()
efectiva = df_transformado["efectividad"].eq("Efectiva")
marca2_puntaje = df_transformado["marca_mate4b_rbd"].eq(2) & con_puntaje

escenarios = []
for nombre, mascara in {
    "A_Efectivos": efectiva & con_puntaje,
    "B_Todos_con_puntaje": con_puntaje,
    "C_Efectivos_mas_marca2": (efectiva | marca2_puntaje) & con_puntaje,
}.items():
    serie = df_transformado.loc[mascara, "prom_mate4b_rbd"]
    escenarios.append({"escenario": nombre, "registros": len(serie), "promedio": serie.mean(), "mediana": serie.median()})

sensibilidad = pd.DataFrame(escenarios)
display(sensibilidad)
print("Marca 2 con puntaje:", int(marca2_puntaje.sum()))
print("Diferencia B - A:", sensibilidad.loc[1, "promedio"] - sensibilidad.loc[0, "promedio"])

,escenario,registros,promedio,mediana
0,A_Efectivos,6524,254.697118,254.0
1,B_Todos_con_puntaje,6579,254.734762,254.0
2,C_Efectivos_mas_marca2,6579,254.734762,254.0


Marca 2 con puntaje: 55
Diferencia B - A: 0.03764378959155579


In [44]:
regional = (
    df_transformado.loc[con_puntaje]
    .groupby("region")
    .apply(lambda g: pd.Series({
        "promedio_todos": g["prom_mate4b_rbd"].mean(),
        "promedio_efectivos": g.loc[g["efectividad"].eq("Efectiva"), "prom_mate4b_rbd"].mean(),
    }), include_groups=False)
)
regional["diferencia"] = regional["promedio_todos"] - regional["promedio_efectivos"]
regional["diferencia_abs"] = regional["diferencia"].abs()
display(regional.sort_values("diferencia_abs", ascending=False))
print("Mayor diferencia regional absoluta:", regional["diferencia_abs"].max())

,promedio_todos,promedio_efectivos,diferencia,diferencia_abs
region,,,,
Arica y Parinacota,258.385714,258.000000,0.385714,0.385714
Los Ríos,247.299213,247.051793,0.247420,0.247420
Libertador General Bernardo O'Higgins,255.502347,255.358670,0.143678,0.143678
Biobío,256.986555,256.874150,0.112405,0.112405
Metropolitana de Santiago,259.008626,259.118398,-0.109772,0.109772
Ñuble,260.612648,260.513944,0.098704,0.098704
Los Lagos,252.559118,252.469636,0.089483,0.089483
Coquimbo,254.489305,254.416667,0.072638,0.072638
Maule,259.199609,259.130693,0.068916,0.068916


Mayor diferencia regional absoluta: 0.38571428571430033


## 10. Normalización y escalamiento: decisión con evidencia

La pauta pide casting/normalización cuando corresponda, no aplicar escalamiento de manera automática. En este núcleo las operaciones son reglas lógicas, mapeos, joins y agregaciones; ninguna depende de distancias o gradientes. Escalar `puntaje_promedio` o `n_alumnos` reduciría interpretabilidad sin mejorar estas operaciones.

Por ello F3 **conserva las unidades originales**. Si una fase posterior usa KNN, K-Means, SVM, redes neuronales u otro método sensible a escala, el escalamiento deberá incorporarse como un transformador ajustado únicamente con el conjunto de entrenamiento.

In [45]:
distribuciones = df_transformado[["prom_mate4b_rbd", "nalu_mate4b_rbd"]].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]
).T
display(distribuciones)
print("Decisión F3: no escalar variables numéricas en el núcleo actual.")

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
prom_mate4b_rbd,6579.0,254.734762,25.615483,152.0,192.0,214.0,237.0,254.0,272.0,297.0,313.00,341.0
nalu_mate4b_rbd,7143.0,29.954781,29.617364,0.0,0.0,1.0,8.0,22.0,41.0,84.9,133.58,273.0


Decisión F3: no escalar variables numéricas en el núcleo actual.


## 11. Documentación de arquitectura y trazabilidad

La arquitectura se genera desde las clases reales para reducir el riesgo de que el informe describa componentes que ya no existen. El repositorio debe conservar `F2/`, `F3/`, `src/`, `tests/`, `docs/` y `data/processed/`.

In [46]:
display(tabla_arquitectura())
print("Archivos F3 clave:")
for ruta in [
    PROJECT_ROOT / "src" / "poo.py",
    PROJECT_ROOT / "src" / "algoritmos.py",
    PROJECT_ROOT / "src" / "arquitectura.py",
    PROJECT_ROOT / "tests" / "test_f3.py",
    PROJECT_ROOT / "docs" / "arquitectura_f3.md",
]:
    print("OK" if ruta.exists() else "FALTA", "-", ruta.relative_to(PROJECT_ROOT))

,clase,hereda_de,responsabilidad,metodos_publicos,modulo
0,Transformador,ABC,Contrato común para transformaciones del pipel...,transformar,src/poo.py
1,TransformadorTipos,Transformador,Contrato común para transformaciones del pipel...,transformar,src/poo.py
2,TransformadorTextos,Transformador,Contrato común para transformaciones del pipel...,transformar,src/poo.py
3,TransformadorCategorias,Transformador,Contrato común para transformaciones del pipel...,transformar,src/poo.py
4,TransformadorGeografia,Transformador,Encapsula la dimensión geográfica necesaria pa...,transformar,src/poo.py
5,TransformadorEfectividad,Transformador,Contrato común para transformaciones del pipel...,transformar,src/poo.py
6,PipelineSIMCE,object,Compone transformadores con bajo acoplamiento ...,"construir_producto_final, ejecutar",src/poo.py


Archivos F3 clave:
OK - src\poo.py
FALTA - src\algoritmos.py
OK - src\arquitectura.py
FALTA - tests\test_f3.py
FALTA - docs\arquitectura_f3.md


## 12. Verificación final de entrega

Esta celda actúa como contrato mínimo de F3. Si falla, el notebook no debe considerarse listo para entregar.

In [47]:
checks = {
    "Fuente cargada": len(df_raw) > 0,
    "346 comunas": len(dim_geo) == 346,
    "Producto final no vacío": not df_final.empty,
    "Continuidad 6524 efectivos": len(df_final) == 6524,
    "619 excluidos auditados": len(auditoria) == 619,
    "Jerarquía recursiva válida": errores_geo == [],
    "Suite de tests OK": resultado_pruebas.returncode == 0,
    "Arquitectura documentada": (PROJECT_ROOT / "docs" / "arquitectura_f3.md").exists(),
}
display(pd.DataFrame(checks.items(), columns=["verificacion", "cumple"]))
assert all(checks.values()), "F3 contiene verificaciones pendientes."
print("F3 verificada: notebook reproducible y núcleo modular operativo.")

,verificacion,cumple
0,Fuente cargada,True
1,346 comunas,True
2,Producto final no vacío,True
3,Continuidad 6524 efectivos,True
4,619 excluidos auditados,True
5,Jerarquía recursiva válida,True
6,Suite de tests OK,True
7,Arquitectura documentada,False


AssertionError: F3 contiene verificaciones pendientes.